Helper Functions

In [ ]:
#Imports
from tqdm import tqdm
import pandas as pd
import numpy as np
from dateutil.rrule import rrule, MONTHLY, DAILY
from scipy.stats import multivariate_normal, invwishart
from numpy.linalg import cholesky
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
import sys
import seaborn as sns
from copy import copy
import pickle
from sklearn.covariance import LedoitWolf
from datetime import timedelta, datetime
import random
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
import glasbey

In [ ]:
#Data clean
def clean_data(data,pct):
    if pct < 1:
        #Choose top pct countries
        agg_data = data[['src_ref','dst_ref','commodity_ref','quantity_kg','value_usd']]
        agg_data = agg_data.groupby(['src_ref','dst_ref','commodity_ref']).sum().reset_index()
        agg_data_2603 = agg_data[agg_data['commodity_ref']==2603].copy()
        agg_data_7402 = agg_data[agg_data['commodity_ref']==7402].copy()
        agg_data_7403 = agg_data[agg_data['commodity_ref']==7403].copy()
        agg_data_7408 = agg_data[agg_data['commodity_ref']==7408].copy()
        agg_data_7408['percent'] = agg_data_7408['quantity_kg'] / agg_data_7408['quantity_kg'].sum()
        agg_data_7408 = agg_data_7408.sort_values('percent',ascending=[False]).reset_index()
        agg_data_7408['cum_pct'] = np.cumsum(agg_data_7408['percent'])
        id7408 = (agg_data_7408['cum_pct'] >= pct).idxmax()
        pct_7408 = agg_data_7408['cum_pct'][id7408]
        country_list_7408 = list(agg_data_7408['src_ref'][0:id7408+1])

        suppliers_7403 = {}
        for country in country_list_7408:
            country_data = agg_data_7403[agg_data_7403['dst_ref'] == country].copy()
            country_data['percent'] = country_data['quantity_kg'] / country_data['quantity_kg'].sum()
            country_data = country_data.sort_values('percent',ascending=[False]).reset_index()
            country_data['cum_pct'] = np.cumsum(country_data['percent'])

            id_c = (country_data['cum_pct'] >= pct).idxmax()
            pct_7403 = country_data['cum_pct'][id_c]
            country_list_c = list(country_data['src_ref'][0:id_c+1])
            suppliers_7403[country] = country_list_c
        country_list_7403 = list({item for sublist in suppliers_7403.values() for item in sublist})

        suppliers_7402 = {}
        for country in country_list_7403:
            country_data = agg_data_7402[agg_data_7402['dst_ref'] == country].copy()
            country_data['percent'] = country_data['quantity_kg'] / country_data['quantity_kg'].sum()
            country_data = country_data.sort_values('percent',ascending=[False]).reset_index()
            country_data['cum_pct'] = np.cumsum(country_data['percent'])

            id_c = (country_data['cum_pct'] >= pct).idxmax()
            pct_7402 = country_data['cum_pct'][id_c]
            country_list_c = list(country_data['src_ref'][0:id_c+1])
            suppliers_7402[country] = country_list_c
        country_list_7402 = list({item for sublist in suppliers_7402.values() for item in sublist})

        suppliers_2603 = {}
        for country in country_list_7402:
            country_data = agg_data_2603[agg_data_2603['dst_ref'] == country].copy()
            country_data['percent'] = country_data['quantity_kg'] / country_data['quantity_kg'].sum()
            country_data = country_data.sort_values('percent',ascending=[False]).reset_index()
            country_data['cum_pct'] = np.cumsum(country_data['percent'])
            if len(country_data) > 0:
                id_c = (country_data['cum_pct'] >= pct).idxmax()
                pct_2603 = country_data['cum_pct'][id_c]
                country_list_c = list(country_data['src_ref'][0:id_c+1])
                suppliers_2603[country] = country_list_c
        #country_list_2603 = list({item for sublist in suppliers_2603.values() for item in sublist})

        combined_data_7408 = pd.DataFrame()
        for country in country_list_7408:
            data_7408_c = data[(data['src_ref'] == country) & (data['commodity_ref'] == 7408)]
            combined_data_7408 = pd.concat([combined_data_7408,data_7408_c],ignore_index=True)

        combined_data_7403 = pd.DataFrame()
        for dst in suppliers_7403:
            supplier_list = suppliers_7403[dst]
            for supplier in supplier_list:
                data_7403_s = data[(data['src_ref'] == supplier) & (data['dst_ref'] == dst) & (data['commodity_ref'] == 7403)]
                combined_data_7403 = pd.concat([combined_data_7403,data_7403_s],ignore_index=True)

        combined_data_7402 = pd.DataFrame()
        for dst in suppliers_7402:
            supplier_list = suppliers_7402[dst]
            for supplier in supplier_list:
                data_7402_s = data[(data['src_ref'] == supplier) & (data['dst_ref'] == dst) & (data['commodity_ref'] == 7402)]
                combined_data_7402 = pd.concat([combined_data_7402,data_7402_s],ignore_index=True)

        combined_data_2603 = pd.DataFrame()
        for dst in suppliers_2603:
            supplier_list = suppliers_2603[dst]
            for supplier in supplier_list:
                data_2603_s = data[(data['src_ref'] == supplier) & (data['dst_ref'] == dst) & (data['commodity_ref'] == 2603)]
                combined_data_2603 = pd.concat([combined_data_2603,data_2603_s],ignore_index=True)

        data = pd.concat([combined_data_2603,combined_data_7402,combined_data_7403,combined_data_7408],ignore_index=True)

    data['price'] = data['value_usd']/data['quantity_kg']
    data['year_month_day'] = data['start_time'].dt.tz_localize(None).dt.to_period('D')
    minday = data['year_month_day'].min()
    data['starttimestep'] = (data['year_month_day'] - minday).apply(lambda x: x.n) + 1
    data['endtimestep'] = data['starttimestep'] + 1
    data = data.drop(columns=['year_month_day'])
    #data = data.sample(n=200).reset_index(drop=True)
    return data

In [ ]:
#Add reserves
def add_reserves(data):
    hs_code_list=[2603,7402,7403,7408]
    hs_code_list = pd.DataFrame(hs_code_list)
    # Convert hs_code_list to a dictionary for faster lookups
    hscode_to_index = {hscode: i for i, hscode in enumerate(hs_code_list[0])}
    
    data['quantity_kg'] = data['quantity_kg'].astype('float')
    data['value_usd'] = data['value_usd'].astype('float')

    # Create node index mapping once
    all_nodes = pd.unique(data[['src_ref', 'dst_ref']].values.ravel('K'))
    node_to_index = {node: i for i, node in enumerate(all_nodes)}
    
    M = len(all_nodes)
    T = int(max(data['endtimestep']))

    timestep_to_date = data.groupby('starttimestep')['start_time'].min().to_dict()

    # Pre-allocate arrays for better performance
    #current_values = np.zeros((M, len(hs_code_list)))
    current_quantities = np.zeros((M, len(hs_code_list)))
    
    # Use a list to collect results, which is more efficient for appending
    new_data_list = []
    
    # Pre-compute data indices by timestep to avoid repeated filtering
    shipments_by_time = {i+1: data[data['starttimestep'] == i+1].index.to_list() 
                         for i in range(T) if any(data['starttimestep'] == i+1)}
    
    # Track arrivals more efficiently
    arrivals_by_time = {i+1: data[data['endtimestep'] == i+1].index.to_list() 
                         for i in range(T) if any(data['endtimestep'] == i+1)}
    
    I2603 = data[data['commodity_ref'] == 2603].index
    P2603 = np.mean(data['price'][I2603])
    I7402 = data[data['commodity_ref'] == 7402].index
    P7402 = np.mean(data['price'][I7402])
    I7403 = data[data['commodity_ref'] == 7403].index
    P7403 = np.mean(data['price'][I7403])
    I7408 = data[data['commodity_ref'] == 7408].index
    P7408 = np.mean(data['price'][I7408])
    
    for i in range(T):
        # Process arrivals at current timestep
        if i+1 in arrivals_by_time:
            for A in arrivals_by_time[i+1]:
                row = data.iloc[A]
                RN = row['dst_ref']
                Q = row['quantity_kg']
                V = row['value_usd']
                H = row['commodity_ref']
                FR = node_to_index[RN]
                c = hscode_to_index[H]
                current_quantities[FR, c] += Q
                #current_values[FR, c] += V
        
        # Process shipments at current timestep
        if i+1 in shipments_by_time:
            for S in shipments_by_time[i+1]:
                row = data.iloc[S]
                SN = row['src_ref']
                Q = row['quantity_kg']
                V = row['value_usd']
                HR = row['commodity_ref']
                
                FS = node_to_index[SN]
                c = hscode_to_index.get(HR, None)
                
                if c == 0:
                    current_quantities[FS,0] -= Q
                    #current_values[FS,0] -= V
                else:
                    H = hs_code_list[0][c-1]

                    test_Q = current_quantities[FS,c-1] - Q
                    current_quantities[FS,c-1] -= Q
                    #test_V = current_values[FS,c-1] - V
                    #current_values[FS,c-1] -= V

                    if test_Q < 0:
                        current_quantities[FS,c-1] -= test_Q
                        #current_values[FS,c-1] -= test_V

                        start_t = i
                        end_t = i + 1
                        
                        # Find start date for timestep `i` (which is index `i`)
                        date_i = timestep_to_date.get(start_t, None)
                        if date_i is None:
                            # Fallback if the dict lookup fails
                            base_date = min(timestep_to_date.values())
                            date_i = base_date + timedelta(days=int(start_t))

                        date_i_minus_1 = date_i.replace(day=1)
                        date_i_minus_1 = date_i_minus_1 - timedelta(days=1)
                        date_i_minus_1 = date_i_minus_1.replace(day=1)
                        date_i_minus_2 = date_i_minus_1 - timedelta(days=1)
                        date_i_minus_2 = date_i_minus_2.replace(day=1)
                        date_i_minus_3 = date_i_minus_2 - timedelta(days=1)
                        date_i_minus_3 = date_i_minus_3.replace(day=1)

                        if H == 2603:
                            transaction_2603 = ['R'+SN, SN, date_i_minus_1, date_i_minus_1, 2603, -test_Q, -P2603*test_Q, P2603, start_t, end_t]
                            new_data_list.append(transaction_2603)
                        elif H == 7402:
                            transaction_2603 = ['R'+SN, 'R'+SN, date_i_minus_2, date_i_minus_2, 2603, -test_Q, -P2603*test_Q, P2603, start_t, end_t]
                            transaction_7402 = ['R'+SN, SN, date_i_minus_1, date_i_minus_1, 7402, -test_Q, -P7402*test_Q, P7402, start_t, end_t]
                            new_data_list.append(transaction_2603)
                            new_data_list.append(transaction_7402)
                        elif H == 7403:
                            transaction_2603 = ['R'+SN, 'R'+SN, date_i_minus_3, date_i_minus_3, 2603, -test_Q, -P2603*test_Q, P2603, start_t, end_t]
                            transaction_7402 = ['R'+SN, 'R'+SN, date_i_minus_2, date_i_minus_2, 7402, -test_Q, -P7402*test_Q, P7402, start_t, end_t]
                            transaction_7403 = ['R'+SN, SN, date_i_minus_1, date_i_minus_1, 7403, -test_Q, -P7403*test_Q, P7403, start_t, end_t]
                            new_data_list.append(transaction_2603)
                            new_data_list.append(transaction_7402)
                            new_data_list.append(transaction_7403)
    
    # Convert collected data to DataFrame at the end (more efficient)
    columns = ['src_ref', 'dst_ref', 'start_time', 'end_time', 
               'commodity_ref', 'quantity_kg', 'value_usd', 'price', 
               'starttimestep', 'endtimestep']
    new_data = pd.DataFrame(new_data_list, columns=columns)
    new_data = new_data.groupby(['src_ref', 'dst_ref', 'commodity_ref', 'start_time', 'end_time', 'price', 'starttimestep', 'endtimestep']).agg({'quantity_kg':'sum','value_usd':'sum'}).reset_index()
    new_data = pd.concat([data,new_data.astype(data.dtypes)])

    # Sort once at the end instead of twice
    new_data = new_data.sort_values(['starttimestep', 'endtimestep']).reset_index(drop=True)

    return new_data

In [ ]:
#Read and format data (CSV)
def read_and_format_data_csv(file,var_2_track,pct,time_int):

    #Open file
    data = pd.read_csv(file,parse_dates=['start_time','end_time'])
    data = clean_data(data,pct)
    #data = add_reserves(data)

    if time_int == DAILY:
        period = 'D'
    elif time_int == MONTHLY:
        period = 'M'
        
    data['period'] = data['start_time'].dt.tz_localize(None).dt.to_period(period)
    minperiod = data['period'].min()
    data = data.drop(columns=['starttimestep','endtimestep'])
    data['starttimestep'] = (data['period'] - minperiod).apply(lambda x: x.n) + 1
    data['endtimestep'] = data['starttimestep'] + 1
    data = data.drop(columns=['period'])
    
    #Create dictionary of which entity supplies which entity
    dst_to_source={}
    for transaction_id, transaction_info in tqdm(data.iterrows()):

        if transaction_info['commodity_ref'] == 2603:
            string1 = '[0]'
            string2 = '[1]'
        elif transaction_info['commodity_ref'] == 7402:
            string1 = '[1]'
            string2 = '[2]'
        elif transaction_info['commodity_ref'] == 7403:
            string1 = '[2]'
            string2 = '[3]'
        elif transaction_info['commodity_ref'] == 7408:
            string1 = '[3]'
            string2 = '[4]'

        destination=transaction_info['dst_ref'] + string2
        source=transaction_info['src_ref'] + string1 
        variable = transaction_info[var_2_track]
        starttime = transaction_info['start_time']
        endtime = transaction_info['end_time']
        transaction_data=pd.DataFrame({var_2_track:variable,'starttime':starttime,'endtime':endtime}, index = [transaction_id])
        sources_dict=dst_to_source.get(destination,{})
        
        if source in sources_dict:
            sources_dict[source] = pd.concat([sources_dict[source],transaction_data])
        else:
            sources_dict[source]=transaction_data

        dst_to_source[destination]=sources_dict
    
    #Determine times of all transactions
    times = [data.loc[id,'start_time'] for id in data.index]
    
    #Determine number of months transaction list spans over
    max_time = max(times)
    min_time = min(times)
    start_date = datetime(min_time.year, min_time.month, 1, tzinfo=min_time.tzinfo)
    time_int_list = list(rrule(time_int, dtstart=start_date, until=max_time))
    if time_int == MONTHLY:
        time_int_list = [dt.replace(day=1) for dt in time_int_list]

    mapping = {
        2603: ('[0]', '[1]'),
        7402: ('[1]', '[2]'),
        7403: ('[2]', '[3]'),
        7408: ('[3]', '[4]')
    }

    strings = data['commodity_ref'].map(mapping)
    string1 = strings.str[0].fillna('')
    string2 = strings.str[1].fillna('')

    src_series = data['src_ref'].astype(str) + string1
    dst_series = data['dst_ref'].astype(str) + string2

    #All edge data
    data['src_ref'] = [val for val in src_series]
    data['dst_ref'] = [val for val in dst_series]
    all_edges = data.groupby(['src_ref', 'dst_ref', 'commodity_ref']).agg({var_2_track:'sum'}).reset_index()
    all_edges = all_edges[['src_ref','dst_ref','commodity_ref']]
    all_edges['change'] = 0.0

    data['src_ref'] = [{'id': val} for val in src_series]
    data['dst_ref'] = [{'id': val} for val in dst_series]
    data['commodity_ref'] = [{'id': str(val)} for val in data['commodity_ref']]

    data = data.to_dict(orient='index')

    return data, dst_to_source, all_edges, time_int_list    

In [ ]:
#Construct tiers
def construct_tiers(end_node,dst_to_source):
    tier_list={'End Node':end_node}
    tier_list['Tier 1']=list(dst_to_source[end_node].keys())
    n=2
    seen_ids=[end_node]
    while len(tier_list[f'Tier {n-1}'])>0:
        tier_list[f'Tier {n}']=[]
        for facility_id in tier_list[f'Tier {n-1}']:
            tier_list[f'Tier {n}']=tier_list[f'Tier {n}']+list(dst_to_source.get(facility_id,{}).keys())
        tier_list[f'Tier {n}']=set(tier_list[f'Tier {n}'])
        seen_ids=list(seen_ids)+list(tier_list[f'Tier {n-1}'])
        tier_list[f'Tier {n}']=list(set(tier_list[f'Tier {n}'])-set(seen_ids)) #added list function
        n+=1    
    tier_list.pop(f'Tier {n-1}')
    return tier_list

In [ ]:
#Minor helper functions
# Adapted from the Reverse Stress Testing repository by Arcady Novosyolov
# (accompanying the work by Kopeliovich et al.)
# Source: https://github.com/arcadynovosyolov/reverse_stress_testing

def gram_schmidt_columns(x):
    ''' column-wise Gram Schmidt procedure '''
    q, r = np.linalg.qr(x)
    return q

def gram_schmidt_rows(x):
    ''' row-wise Gram Schmidt procedure '''
    q, r = np.linalg.qr(x.T)
    return q.T

def p_from_w(w):
    ''' given weights vector w, calculate transform matrix p, as described in Appendix A of the paper'''
    w = np.array(w)
    n = len(w)
    p = np.zeros((n, n))
    p[0, :] = w / np.linalg.norm(w)
    j0 = (np.abs(w) > 0.00001).nonzero()[0][0]
    for j in range(j0):
        p[j + 1, j] = 1
    for j in range(j0 + 1, n):
        p[j, j] = 1
    gs = gram_schmidt_rows(p)
    for j in range(gs.shape[1]):
        if gs[0, j] < 0:
            gs[:, j] = -gs[:, j]
    return gs

def standard_simplex_vertices(M):
    ''' just that '''
    z = np.eye(M) - 1 / M
    a = np.eye(M)
    a[0, :] = 1
    p = gram_schmidt_rows(a)
    cls = np.append(range(1, M), 0)
    gs = np.delete(p.dot(z), 0, axis=0)[:, cls]
    if gs[0, 0] < 0:
        gs = -gs
    return gs * np.sqrt(M / (M - 1))

def quadratic_form_inv(a, x):
    ''' x * a^(-1) * x '''
    ax = np.linalg.solve(a, x)
    return x.T.dot(ax)

def make_projection_matrix(cvm):
    ''' makes matrix that projects the distribution onto the hyperplane'''
    w = np.ones(len(cvm))

    # prepare transform matrix
    p = p_from_w(w)
    n = len(w)
    
    # compute covariance matrix in the transformed space
    dy = p.dot(cvm.dot(p.T))
    
    # extract decomposition parts
    d11 = dy[0, 0]
    # d1I = dy[0, 1:]
    dI1 = dy[1:, 0]
    dII = dy[1:, 1:]

    # compute center and conditional distribution in transformed space
    i_m = np.matrix(dI1).reshape(n - 1, 1) * np.matrix(dI1).reshape(1, n - 1)
    dyh = dII - i_m / d11

    return dyh

In [ ]:
#Check if covariance matrix is SPD
def check_ns_pd(cvm):
    n = len(cvm)
    try:
        np.linalg.inv(cvm)
        multivariate_normal.logpdf(np.zeros(n), mean=np.zeros(n), cov=cvm)
        eigens = np.linalg.eigh(cvm)
        vals = eigens.eigenvalues
        _ = cholesky(cvm)
        first_check = True
    except:
        first_check = False

    if first_check:
        check_inf = np.isinf(multivariate_normal.logpdf(np.zeros(n), mean=np.zeros(n), cov=cvm))
        if np.min(vals) <= 0 or np.iscomplexobj(vals) or check_inf:
            final_check = False
        else:
            final_check = True
    else:
        final_check = False

    return final_check

In [ ]:
#Ensure covariance matrix and its projection matrix are both nonsingular and symmetric positive definite
def make_matrix_ns_pd(cvm):
    ind = cvm.columns
    x = -16
    cvm = np.array(cvm,dtype=float)
    copy_cvm = copy(cvm)
    check1 = False
    check2 = False

    while check1 == False or check2 == False:
      
        vals, vecs = np.linalg.eigh(copy_cvm)
        vals_clipped = np.maximum(vals, 10**x)
        cvm = (vecs * vals_clipped) @ vecs.T
        
        cvm = (cvm + cvm.T)/2
        check1 = check_ns_pd(cvm)
        if len(cvm) == 1:
            check2 = True
        else:
            p = make_projection_matrix(cvm)
            check2 = check_ns_pd(p)
    
        x += 1

    cvm = pd.DataFrame(cvm,columns=ind,index=ind)
    return cvm

In [ ]:
#Covariance matrix
def find_covariance_matrix(end_node,time_int_list,data,var_2_track,keep_zeros,reliable_thresh=0.0):
    time_int_list2 = pd.DataFrame(time_int_list, columns=[0])
    data_df=pd.DataFrame(data).T
    data_df['src_ref']=data_df['src_ref'].str['id']
    data_df['dst_ref']=data_df['dst_ref'].str['id']
    data_df['start_time']=pd.to_datetime(data_df['start_time'])
    data_df['end_time']=pd.to_datetime(data_df['end_time'])
    data_df['start_t']=data_df['start_time'].dt.year.astype(str)+data_df['start_time'].dt.month.astype(str).str.zfill(2)+data_df['start_time'].dt.day.astype(str).str.zfill(2)+data_df['start_time'].dt.hour.astype(str).str.zfill(2)
    
    filtered=data_df[data_df['dst_ref']== end_node]
    filtered = filtered[['src_ref',var_2_track,'start_t']]
    filtered = filtered.groupby(['start_t','src_ref']).sum()
    with pd.option_context('future.no_silent_downcasting', True):
        pivoted_df = filtered.pivot_table(index='start_t',columns='src_ref',values=var_2_track,fill_value=0)
    
    non_zero_proportion = (pivoted_df != 0).sum() / len(pivoted_df.index)
    cols_to_keep = non_zero_proportion[non_zero_proportion > reliable_thresh].index.tolist()
    dropped_cols = list(set(pivoted_df.columns).symmetric_difference(set(cols_to_keep)))
    pivoted_df = pivoted_df[cols_to_keep]

    if keep_zeros:
        time_int_list3 = time_int_list2[0].dt.year.astype(str)+time_int_list2[0].dt.month.astype(str).str.zfill(2)+time_int_list2[0].dt.day.astype(str).str.zfill(2)+time_int_list2[0].dt.hour.astype(str).str.zfill(2)
        final_df = pivoted_df.reindex(time_int_list3,fill_value=0)
    else: 
        final_df = pivoted_df

    means = final_df.mean(0)
    means = means[means != 0]

    if final_df.shape[1] == 0:
        cov_matrix = final_df.cov()
    else:
        lw = LedoitWolf().fit(final_df)
        cov_matrix_ar = lw.covariance_
        cov_matrix = pd.DataFrame(cov_matrix_ar,index=cols_to_keep,columns=cols_to_keep).astype('float')

    cov_matrix = cov_matrix.loc[(cov_matrix != 0).any(axis=1)]
    cov_matrix = cov_matrix.loc[:, (cov_matrix != 0).any(axis=0)]
    if len(cov_matrix) > 0:
        NSSPD1 = check_ns_pd(cov_matrix)
        if len(cov_matrix) == 1:
            NSSPD2 = True
        else:
            p = make_projection_matrix(cov_matrix)
            NSSPD2 = check_ns_pd(p)
        
        if not NSSPD1 or not NSSPD2:
            cov_matrix = make_matrix_ns_pd(cov_matrix)

    return cov_matrix, means, dropped_cols

In [ ]:
#Calculate base changes
def find_base_scenario(L,D,means):
    w = np.ones((len(D)))
    base_scenario = (L*D.dot(w))/w.dot(D.dot(w))
    '''
    #OOB constraint
    test = means + base_scenario
    if np.any(test<0):    
        extra = -np.sum(test[test<0])
        test[test<0] = 0.0
        if np.sum(test) == 0.0:
            base_scenario = pd.Series(-means.to_numpy(), index=means.index, dtype=float)
        else:
            ws = test/np.sum(test)
            base_scenario = test - ws*extra - means
            base_scenario = np.maximum(base_scenario,-means)
    '''
    
    loglik = multivariate_normal.logpdf(base_scenario, mean=np.zeros((len(D),)), cov=D)
    if loglik == np.inf:
        print('log(lik) is infinity.')
    
    if loglik == -np.inf:
        print('Probability density of base scenario is 0. Forcing probability of base scenario to 1.')
        
    return base_scenario

In [ ]:
#Tier i to Tier i+1 base change given a series of changes on Tier i
def calculate_next_tier_base_changes(all_covs,all_means,all_edges,dst_to_source,tier_list,i,current_change,skip_thresh):
    next_change = pd.Series([],dtype='float64')
    tier_nodes = list(tier_list[f'Tier {i}'])
    for node in tier_nodes:
        if node in dst_to_source and node in current_change.index:
            cov_matrix = all_covs[node]['0']
            means = all_means[node]
            supply_nodes = list(cov_matrix.columns)
            if len(cov_matrix) > 0 and abs(current_change[node]) >= skip_thresh:
                L = current_change[node]
                change = find_base_scenario(L,cov_matrix,means)
                next_change = next_change.add(change, fill_value=0)

                for supply in supply_nodes:
                    ind = all_edges[(all_edges['src_ref'] == supply) & (all_edges['dst_ref'] == node)].index
                    all_edges.loc[ind,'change'] += change[supply] 

    return next_change, all_edges

In [ ]:
#Plot base scenarios
def plot_base_scenarios(all_edges,end_user,P):
    
    G = nx.from_pandas_edgelist(all_edges,source='src_ref',target='dst_ref',create_using=nx.DiGraph)
    ancestors = sorted(nx.ancestors(G,end_user))
    edges_options = all_edges[(all_edges['src_ref'].isin(ancestors)) & (all_edges['dst_ref'].isin(ancestors))][['src_ref','dst_ref']].to_numpy()
    end_user_edges = all_edges[(all_edges['dst_ref']==end_user)][['src_ref','dst_ref']].to_numpy()
    edges_options = np.vstack((edges_options,end_user_edges))
        
    #Don't repeat edges in final edge list 
    edges = [tuple(x) for x in edges_options]
    if len(edges) == 0:
        print('No graph was created because there is no path') 
        sys.exit(0)

    H = nx.from_edgelist(edges,create_using=nx.DiGraph) 

    #assigning percent change in variable as node attribute
    source_node_outputs = all_edges
    for edge in edges:
        H.edges[edge]['change'] = source_node_outputs[(source_node_outputs['src_ref']==edge[0]) & (source_node_outputs['dst_ref']==edge[1])].iloc[0]['change']

    # Create figure and axis explicitly
    fig, ax = plt.subplots(figsize=(40, 40))
      
    # Create layered positions based on distance from end_user
    pos = {}
    layers = {}
    
    # First, determine each node's layer based on longest path distance to end_user
    for node in H.nodes():
        # Try to find shortest path to end_user
        try:
            path = nx.shortest_path(H, node, end_user)
            distance = len(path) - 1  # Distance to end_user
            if distance not in layers:
                layers[distance] = []
            layers[distance].append(node)
        except nx.NetworkXNoPath:
            # If no path to end_user, place at far left
            if -1 not in layers:
                layers[-1] = []
            layers[-1].append(node)
    
    # Sort layers by distance (highest first)
    sorted_layers = sorted(layers.keys(), reverse=True)
    
    # Assign x-coordinates by layer (distance from end_user)
    max_layer = max(sorted_layers) if sorted_layers else 0
    for layer in sorted_layers:
        nodes_in_layer = layers[layer]
        x_pos = (max_layer - layer) / max(1, max_layer)  # Normalize to 0-1
        
        # Distribute nodes in this layer vertically
        for i, node in enumerate(nodes_in_layer):
            y_pos = (i - len(nodes_in_layer)/2) / max(1, len(nodes_in_layer))
            pos[node] = (x_pos, y_pos)
    
    # Ensure end_user is at far right
    if end_user in pos:
        pos[end_user] = (1.0, 0.0)
    
    # Get 'change' values for node coloring
    diff_values = [H.edges[edge].get('change', 0) for edge in H.edges()]
    
    # Create colormap with normalization
    vmin = min(diff_values)
    vmax = max(diff_values)
    rnge = max(abs(vmin),abs(vmax))
    norm = plt.Normalize(vmin=-rnge, vmax=rnge)
    cmap = cm.coolwarm.reversed()  # Blue to red colormap
    mapper = cm.ScalarMappable(norm=norm, cmap=cmap)
    # Draw the graph
    nodes = nx.draw_networkx_nodes(
        H, pos,
        node_size=1500,
        node_color='lightgray',
        edgecolors='black',
        linewidths=2.5,
        ax=ax  # Pass the axis explicitly
    )
    edge_colors = [mapper.to_rgba(d['change']) for u,v,d in H.edges(data=True)]
    edges = nx.draw_networkx_edges(
        H, pos,
        edge_color=edge_colors,
        width=5.0,
        arrowsize=25,
        arrowstyle='-|>',
        # connectionstyle='arc3,rad=0.1',
        ax=ax  # Pass the axis explicitly
    )
    
    # Add labels with minimal whitespace
    labels = {node: node[:-3] for node in H.nodes()}
    nx.draw_networkx_labels(
        H, pos,
        labels=labels,
        font_size=12,
        font_weight='bold',
        ax=ax  # Pass the axis explicitly
    )
    
    # Add colorbar with explicit axis reference
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, shrink=0.6)
    cbar.set_label('Monthly Base Changes (% of Total Loss)', fontsize=20)
    cbar.ax.tick_params(labelsize=16)
    
    # Set title and remove axis
    ax.set_title(f'RST Output Results Based on a {100*P}% Change: {end_user[:-2]}', fontsize=20)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
#Compute all scenarios and approximate their probabilities and collapse them if appropriate
# Adapted from the Reverse Stress Testing repository by Arcady Novosyolov
# (accompanying the work by Kopeliovich et al.)
# Source: https://github.com/arcadynovosyolov/reverse_stress_testing

def comp_scenarios(c, L, q, means):
    w = np.ones((len(c)))

    # base scenario
    cw = c.dot(w)
    ah = L * cw / w.dot(cw)

    # prepare transform matrix
    p = p_from_w(w)
    n = len(w)

    ahy_full = p.dot(ah)
    first_comp = ahy_full[0]

    # supply nodes
    supply_nodes = c.columns

    # compute covariance matrix in the transformed space
    dy = p.dot(c.dot(p.T))

    # extract decomposition parts
    d11 = dy[0, 0]
    # d1I = dy[0, 1:]
    dI1 = dy[1:, 0]
    dII = dy[1:, 1:]

    # compute center and conditional distribution in transformed space
    wn = np.linalg.norm(w)
    ahy = L * dI1 / d11 / wn
    i_m = np.matrix(dI1).reshape(n - 1, 1) * np.matrix(dI1).reshape(1, n - 1)
    dyh = dII - i_m / d11

    # eigenvalues in ascending order and eigenvectors
    vals, vect = np.linalg.eigh(dyh)
    ind = vals.argsort()[::-1]
    V = vect[:, ind]

    # place the eigenvectors in opposite order
    # V = vect[:, ::-1]

    M = min(len(c), 5)
    # compute vertices of the regular M-simplex
    t = standard_simplex_vertices(M)

    # rotation to principle components space
    z = V[:, :(M - 1)].dot(t)

    # compute the sphere radius
    z_last = z[:, -1]
    denom = quadratic_form_inv(dyh, np.array(z_last))
    r = np.sqrt(-2 * np.log(q) / denom)[0, 0]
    ahyr = ahy.repeat(M).reshape(n - 1, M)
    rz = r * z + ahyr

    # unite central scenario with sphere scenarios (NumPy version)
    df = np.column_stack((ahy, rz))

    # append first_comp row at the beginning
    cls = ['{0}'.format(i) for i in range(M + 1)]
    df = np.vstack((np.full((1, M + 1), first_comp),df))

    # rotating back to initial space
    ar = np.array(p.T.dot(df))
    # for j in range(ar.shape[1]):
    #     ar[:,j] *= w
    '''
    #OOB constraint
    for j in range(ar.shape[1]):
        test = means + ar[:,j]
        if np.any(test<0):
            extra = -np.sum(test[test<0])
            test[test<0] = 0.0
            if np.sum(test) <= 0.0:
                ar[:,j] = -means.to_numpy()
            else:
                ws = test/np.sum(test)
                ar[:,j] = test - ws*extra - means
                ar[:,j] = np.maximum(ar[:,j],-means.to_numpy())
    '''
    # turning to dataframe
    scenarios = pd.DataFrame(ar, columns=cls).set_index(supply_nodes)

    # Remove duplicate columns using NumPy
    #_, unique_idx = np.unique(scenarios.to_numpy(), axis=1, return_index=True)
    #unique_idx = np.sort(unique_idx)

    #scenarios = scenarios.iloc[:, unique_idx]
    num_columns = scenarios.shape[1]
    scenarios.columns = range(num_columns)
    M = scenarios.shape[1]

    zero_mean = np.zeros(n)

    loglik = multivariate_normal.logpdf(scenarios.T,mean=zero_mean,cov=c)
    loglik = np.atleast_1d(loglik)
    if loglik[0] == -np.inf:
        scenarios = pd.DataFrame(scenarios['0'])
        prob = pd.DataFrame(np.ones((1, 1)), index=['Probabilities'])
        print('Probability density of base scenario is 0. Forcing probability of base scenario to 1.')
    else:
        mm = np.max(loglik)
        exps = np.exp(loglik - mm)
        prob = exps / np.sum(exps)
        prob = pd.DataFrame(prob.reshape(1, M), columns=range(num_columns), index=['Probabilities'])

    perm = np.lexsort(scenarios.to_numpy()[::-1])
    scenarios = scenarios.iloc[:, perm]
    prob = prob.iloc[:, perm]

    num_columns = scenarios.shape[1]
    scenarios.columns = range(num_columns)
    prob.columns = range(num_columns)

    return scenarios, prob

In [ ]:
#Bayesian enhancements to help distributions look nicer
def bayesian_enhancements(changes,probs,sample_cov,L,q,means):          
    changes_i, probs_i = comp_scenarios(sample_cov, L, q, means)
    changes_i.index = changes.index
    changes_i.columns = list(range(len(changes.columns),len(changes.columns)+len(changes_i.columns)))
    changes_i.columns = [str(col) for col in changes_i.columns]
    changes = pd.concat([changes,changes_i],axis=1)
    probs_i.columns = changes_i.columns
    probs = pd.concat([probs,probs_i],axis=1)
    return changes, probs

In [ ]:
#Combine and aggregate combinations
def combine_and_aggregate(c1=pd.DataFrame([]), c2=pd.DataFrame([]), p1=pd.DataFrame(index=['Probabilities']), p2=pd.DataFrame(index=['Probabilities'])):
    #Preliminary checks for edge cases
    if c1.empty:
        result_c = c2
        result_p = p2
    elif c2.empty:
        result_c = c1
        result_p = p1
    else:
        # If indices match exactly, skip alignment entirely for speed
        if c1.index.equals(c2.index):
            v1 = c1.values[:, :, np.newaxis]
            v2 = c2.values[:, np.newaxis, :]
            combined = (v1 + v2).reshape(len(c1), -1)
            result_c = pd.DataFrame(combined, index=c1.index)
        else:
            # Fast union of indices
            full_index = c1.index.union(c2.index)

            # Reindex using optimized internal routines rather than .align()
            v1_flat = c1.reindex(full_index, fill_value=0).values
            v2_flat = c2.reindex(full_index, fill_value=0).values

            v1 = v1_flat[:, :, np.newaxis]
            v2 = v2_flat[:, np.newaxis, :]
            combined = (v1 + v2).reshape(len(full_index), -1)
            result_c = pd.DataFrame(combined, index=full_index)

        outer_product_values = np.outer(p1.values.ravel(), p2.values.ravel())
        outer_product_values = outer_product_values.reshape((1, len(p1.columns) * len(p2.columns)))
        result_p = pd.DataFrame(outer_product_values, index=['Probabilities'], columns=list(range(outer_product_values.shape[1])))
      
    perm = np.lexsort(result_c.to_numpy()[::-1])

    result_c = result_c.iloc[:, perm]
    result_p = result_p.iloc[:, perm]


    num_columns = result_c.shape[1]
    result_c.columns = range(num_columns)
    result_p.columns = range(num_columns)

    return result_c, result_p

In [ ]:
#Cluster and reduce combinations
def cluster_and_reduce(result_c, result_p, S):
    if S > 0:
        #global total_pdist, total_linkage, num_calls

        #t0 = time.perf_counter()
        pairwise_dist = pdist(result_c.T, metric='minkowski', p=2)
        pairwise_dist = np.round(pairwise_dist, decimals=8)
        #total_pdist += time.perf_counter() - t0

        #t0 = time.perf_counter()
        Z = linkage(pairwise_dist, method='complete')
        #total_linkage += time.perf_counter() - t0

        #num_calls += 1

        # Cluster the data
        distance_threshold = S*np.mean(Z[:, 2]) # Adjust this multiplier (e.g., 0.25 = more clusters, 0.50 = fewer clusters)
        cluster_labels = fcluster(Z, t=distance_threshold, criterion='distance')
        unique_labels, label_counts = np.unique(cluster_labels, return_counts=True)

        # Reconstruct the reduced scenarios and probabilities using fast NumPy conversion
        reduced_scenarios_list = []
        reduced_prob_list = []
        result_c_values = result_c.values
        result_p_values = result_p.values.ravel()

        for i in unique_labels:
            ind_list = (cluster_labels == i)
            group_s = result_c_values[:, ind_list]
            group_p = result_p_values[ind_list]

            combined_p = np.sum(group_p)
            if combined_p == 0:
                p_weights = np.ones(group_s.shape[1]) / group_s.shape[1]
            else:
                p_weights = group_p / combined_p

            combined_s = group_s @ p_weights

            reduced_scenarios_list.append(combined_s)
            reduced_prob_list.append(combined_p)

        reduced_scenarios = pd.DataFrame(
                            np.column_stack(reduced_scenarios_list),
                            index=result_c.index
                        )
        reduced_prob = pd.DataFrame([reduced_prob_list], index=['Probabilities'], columns=range(len(unique_labels)))
        assert np.isclose(np.sum(reduced_prob,1), 1.0)

        perm = np.lexsort(reduced_scenarios.to_numpy()[::-1])

        reduced_scenarios = reduced_scenarios.iloc[:, perm]
        reduced_prob = reduced_prob.iloc[:, perm]

        num_columns = reduced_scenarios.shape[1]
        reduced_scenarios.columns = range(num_columns)
        reduced_prob.columns = range(num_columns)

        return reduced_scenarios, reduced_prob

    else:
        return result_c, result_p

In [ ]:
#Create all combinations of changes and probabilities between two sets of changes and probabilities
def create_all_combinations(S, c1=pd.DataFrame([]), c2=pd.DataFrame([]), p1=pd.DataFrame(index=['Probabilities']), p2=pd.DataFrame(index=['Probabilities'])):
    result_c, result_p = combine_and_aggregate(c1, c2, p1, p2)

    if p1.empty and p2.empty:
        return result_c, result_p

    if result_c.shape[1] < 2:
        return result_c, result_p

    reduced_scenarios, reduced_prob = cluster_and_reduce(result_c, result_p, S)

    return reduced_scenarios, reduced_prob

In [ ]:
#Tier i to Tier i+1 all changes given a dataframe of changes on Tier i
def calculate_next_tier_all_changes(all_covs,all_means,dst_to_source,tier_list,q,S,i,tier_changes_in,tier_probs_in,skip_thresh,num_cov_samples):
    tier_change_list = []
    tier_prob_list = []
    agg_change_list = []
    agg_prob_list = []

    tier_nodes = sorted(list(tier_list[f'Tier {i}']))
    col = 0

    #Loop through each state in the antecedent layer
    for scen in tqdm(tier_changes_in.columns):
        current_scen = tier_changes_in[scen]
        next_tier_scens_from_current_scen = {}
        tier_probs_out_from_current_scen = {}

        #Run Kop. algorithm on every node in tier based on its given state
        for node in tier_nodes: #tqdm
            if node in dst_to_source and node in current_scen.index:
                cov_matrix = all_covs[node]['0']
                means = all_means[node]
                if len(cov_matrix) > 0 and abs(current_scen[node]) >= skip_thresh:
                    L = current_scen[node]
                    if len(cov_matrix) == 1:
                        #If only one parent node, just take the central scenario
                        change = find_base_scenario(L,cov_matrix,means)
                        change = pd.DataFrame(change,columns=['0'])
                        prob = pd.DataFrame(np.ones((1,1)),index=['Probabilities'],columns=['0'])
                    else:
                        change, prob = comp_scenarios(cov_matrix,L,q,means)
                        if num_cov_samples > 0:
                          #Sample covariance matrices if desired
                            for j in range(num_cov_samples):
                                sample_cov = all_covs[node][f'{j+1}']
                                change, prob = bayesian_enhancements(change,prob,sample_cov,L,q,means)
                            prob /= (num_cov_samples + 1)

                    next_tier_scens_from_current_scen[node] = change
                    tier_probs_out_from_current_scen[node] = prob

        #Get all combinations of changes and probabilities
        outer_product_scenarios = pd.DataFrame([])
        outer_product_probabilities = pd.DataFrame([])
        for node in next_tier_scens_from_current_scen:
            outer_product_scenarios, outer_product_probabilities = create_all_combinations(S[i-1],outer_product_scenarios,next_tier_scens_from_current_scen[node],outer_product_probabilities,tier_probs_out_from_current_scen[node])

        tier_change_list.append(outer_product_scenarios)
        tier_prob_list.append(outer_product_probabilities)
        range_list = list(range(outer_product_scenarios.shape[1]))
        new_cols = [item + col for item in range_list]

        vals = tier_changes_in[scen].to_numpy()
        agg_change_temp = pd.DataFrame(np.repeat(vals[:, None], len(new_cols), axis=1),index=tier_changes_in.index,columns=new_cols)
        agg_change_list.append(agg_change_temp)

        probs = tier_probs_in[scen].to_numpy()
        agg_probs_temp = pd.DataFrame(
            np.repeat(probs[:, None], len(new_cols), axis=1),index=tier_probs_in.index,columns=new_cols)
        agg_prob_list.append(agg_probs_temp)
        col += len(new_cols)

    tier_changes_out = pd.concat(tier_change_list, axis=1)
    tier_probs_out = pd.concat(tier_prob_list, axis=1)
    agg_changes = pd.concat(agg_change_list, axis=1)
    agg_probs = pd.concat(agg_prob_list, axis=1)

    tier_changes_out.columns = list(range(len(tier_changes_out.columns)))
    agg_changes = pd.concat([agg_changes,tier_changes_out])
    tier_probs_out.columns = list(range(len(tier_probs_out.columns)))
    agg_probs *= tier_probs_out

    return tier_changes_out, agg_changes, agg_probs

Driver Function

In [ ]:
#Propagate all scenarios through all tiers
'''Main driver script that propogates all changes through all tiers'''
file = 'copper_data_2018-01-01_2024-12-31_cleaned_us.csv'
dataset = 'comtrade'
end_node = 'USA[4]' #end user name
Ps = [-0.05,-0.25,-0.5] #Desired percent gain or loss
skip_thresh = 0 #If the absolute value of an L is below this threshold, skip the RST
time_int = MONTHLY #Time resolution (HOURLY, DAILY, MONTHLY, YEARLY, etc.)
market_thresh = 0.95 #What % of the market share we filter our data out of
reliable_thresh = 0.0 #Percentage of time intervals a supplier must provide in order to count as a reliable supplier 
keep_zeros = True #Whether or not to remove any time intervals where no suppliers provided any goods
q = 0.5 #Parameter used in specifiying the relative likelihood the 'less likely' scenarios are compared to the base scenario 
num_cov_samples = 0 #Number of covariance perturbations for every RST for Bayesian enhancements (can be 0)
var_2_track = 'quantity_kg' #Which variable to track
loss_factors = [1,1,1] 
S = [1,1,1]

data, dst_to_source, all_edges, time_int_list = read_and_format_data_csv(file,var_2_track,market_thresh,time_int)
tier_list = construct_tiers(end_node,dst_to_source)
cov_matrix, means, drops = find_covariance_matrix(end_node,time_int_list,data,var_2_track,keep_zeros,reliable_thresh)
for drop in drops:
    dst_to_source[end_node].pop(drop)
    tier_list['Tier 1'].remove(drop)
    all_edges.drop(all_edges[(all_edges['src_ref']==drop) & (all_edges['dst_ref']==end_node)].index,inplace=True)

all_edges = all_edges.reset_index(drop=True)

try:
    with open(f'copper_{dataset}_{100*market_thresh}_covs.pkl', 'rb') as f:
        loaded_covs = pickle.load(f)
    all_covs = loaded_covs['all_covs']
    all_means = loaded_covs['all_means']
    print('Covs have been loaded')
except:
    all_covs = {end_node: {'0': cov_matrix}}
    all_means = {end_node: means}
    if num_cov_samples > 0:
        for i in range(num_cov_samples):
            sample_cov = pd.DataFrame(invwishart.rvs(df=len(cov_matrix), scale=cov_matrix.to_numpy()))
            sample_cov.index = cov_matrix.index
            sample_cov.columns = cov_matrix.columns
            NSSPD1 = check_ns_pd(sample_cov)
            p = make_projection_matrix(sample_cov)
            NSSPD2 = check_ns_pd(p)
            if not NSSPD1 or not NSSPD2:
                sample_cov = make_matrix_ns_pd(sample_cov)   
            all_covs[end_node][f'{i+1}'] = sample_cov

    for i in range(len(tier_list)-2):
        tier_nodes = tier_list[f'Tier {i+1}']
        for node in tqdm(tier_nodes):
            all_covs[node] = {}
            all_covs[node]['0'], all_means[node], drops = find_covariance_matrix(node,time_int_list,data,var_2_track,keep_zeros,reliable_thresh)
            for drop in drops:
                dst_to_source[node].pop(drop)
                #if drop in tier_list[f'Tier {i+2}']: #?
                #    tier_list[f'Tier {i+2}'].remove(drop) #?

            if num_cov_samples > 0:
                for j in range(num_cov_samples):
                    if len(all_covs[node]['0']) > 0:
                        sample_cov = pd.DataFrame(invwishart.rvs(df=len(all_covs[node]['0']),scale=all_covs[node]['0'].to_numpy()),index=all_covs[node]['0'].index,columns=all_covs[node]['0'].columns)
                        NSSPD = check_ns_pd(sample_cov)
                        if not NSSPD:
                            sample_cov = make_matrix_ns_pd(sample_cov)
                        all_covs[node][f'{j+1}'] = sample_cov

    save_covs = {'all_covs': all_covs, 'all_means': all_means}
    with open(f'copper_{dataset}_{100*market_thresh}_covs.pkl', 'wb') as f:
        pickle.dump(save_covs, f)

for P in Ps:
    all_edges['change'] = 0.0
    L = P*sum(means)
    change_1 = find_base_scenario(L,cov_matrix,means)
    tier_1_nodes = list(change_1.index)
    for node in tier_1_nodes:
        ind = all_edges[(all_edges['src_ref'] == node) & (all_edges['dst_ref'] == end_node)].index
        all_edges.loc[ind,'change'] += change_1[node]

    all_base_changes = {'Tier 1':change_1}
    t = 1

    current_changes = change_1 #Add input-output factor
    while len(current_changes) > 0 and t < len(tier_list) - 1:
        next_change, all_edges = calculate_next_tier_base_changes(all_covs,all_means,all_edges,dst_to_source,tier_list,t,current_changes,skip_thresh)
        all_base_changes[f'Tier {t+1}'] = loss_factors[t-1]*next_change
        current_changes = loss_factors[t-1]*next_change
        t += 1

    if len(cov_matrix) > 1:
        changes_1, prob_1 = comp_scenarios(cov_matrix, L, q, means)
        if num_cov_samples > 0:
            for i in range(num_cov_samples):
                sample_cov = all_covs[end_node][f'{i+1}']
                changes_1, prob_1 = bayesian_enhancements(changes_1,prob_1,sample_cov,L,q,means)
            prob_1 /= (num_cov_samples + 1)
        assert np.isclose(np.sum(prob_1,1), 1.0)
    else:
        changes_1 = find_base_scenario(L,cov_matrix,means)
        changes_1 = pd.DataFrame(changes_1,columns=['0'])
        prob_1 = pd.DataFrame(np.ones((1,1)),index=['Probabilities'],columns=['0'])
 
    tier_changes = {'Tier 1': changes_1}
    agg_changes = {'Tier 1': changes_1}
    agg_probs = {'Tier 1': prob_1}
    current_agg_changes = changes_1 #Add input-output factor
    current_agg_probs = prob_1
    for i in range(len(tier_list)-2):
        changes_i, agg_changes_i, agg_prob_i = calculate_next_tier_all_changes(all_covs,all_means,dst_to_source,tier_list,q,S,i+1,current_agg_changes,current_agg_probs,skip_thresh,num_cov_samples)
        assert np.isclose(np.sum(agg_prob_i,1), 1.0)
        tier_changes[f'Tier {i+2}'] = changes_i
        agg_changes[f'Tier {i+2}'] = agg_changes_i
        agg_probs[f'Tier {i+2}'] = agg_prob_i
        current_agg_changes = agg_changes_i
        current_agg_probs = agg_prob_i

    data_to_save = {'agg_changes': agg_changes, 'agg_probs': agg_probs, 'tier_list': tier_list, 'tier_changes': tier_changes, 'all_base_changes': all_base_changes, 'all_edges': all_edges}
    filename = f'copper_{dataset}_{abs(100*P)}.pkl'
    with open(filename, 'wb') as f:
        pickle.dump(data_to_save, f)

Plotter

In [ ]:
#Comtrade colors
unique_countries = [
    'Australia', 'Belgium', 'Bolivia (Plurinational State of)', 'Brazil', 
    'Bulgaria', 'Canada', 'Chile', 'China', 'Dem. Rep. of the Congo', 
    'Germany', 'Indonesia', 'Japan', 'Kazakhstan', 'Malaysia', 'Mexico', 
    'Namibia', 'Netherlands', 'Panama', 'Peru', 'Philippines', 'Poland', 
    'Rep. of Korea', 'Russian Federation', 'Serbia', 'Slovakia', 'South Africa', 
    'Spain', 'Türkiye', 'Ukraine', 'USA', 'Uzbekistan', 'Zambia'
]


country_codes =  [
    'AUS', 'BEL', 'BOL', 'BRA', 'BGR', 'CAN', 'CHL', 'CHN', 'COD', 
    'DEU', 'IDN', 'JPN', 'KAZ', 'MYS', 'MEX', 'NAM', 'NLD', 'PAN', 
    'PER', 'PHL', 'POL', 'KOR', 'RUS', 'SRB', 'SVK', 'ZAF', 
    'ESP', 'TUR', 'UKR', 'USA', 'UZB', 'ZMB'
]

qualitative_colors = glasbey.create_palette(
    palette_size=32, 
    colorblind_safe=True, 
    lightness_bounds=(20, 65)
)

random.seed(14) 
random.shuffle(qualitative_colors)

colors = qualitative_colors[:len(unique_countries)]
color_dict_comtrade = dict(zip(unique_countries, zip(colors,country_codes)))
#color_dict_comtrade["Other"] = ((0.5, 0.5, 0.5), 'Other')  # Solid grey for "Other"

In [ ]:
#Plot less likely distributions
def plot_less_likely_distributions(all_tier_changes, all_tier_probs, tier, nodes, ax, color_dict, num_samples):
    for node in nodes:
        if node in all_tier_changes[f'{tier}'].index:
            cs = all_tier_changes[f'{tier}'].loc[node].to_numpy().copy()
            cs *= -1
            cs = np.clip(cs, 0, np.inf)
            cs /= 1e6
            ps = all_tier_probs[f'{tier}'].loc['Probabilities'].to_numpy().copy()
            assert np.isclose(np.sum(ps), 1.0)
            ind = cs.argsort()
            cs = cs[ind]
            ps = ps[ind]
            sample_points = np.random.choice(cs, size=num_samples, p=ps)
            
            # Fetch the color from color_dict using the indexed node name (e.g., 'Canada[3]')
            # Fallback to standard gray (0.5, 0.5, 0.5) if the node isn't found
            node_color = color_dict[node[:-3]][0]            
            if np.var(sample_points) < 1e-7:
                # Force a solid vertical line at the exact value
                ax.axvline(x=sample_points[0], color=node_color, lw=2.5, linestyle='-', label=node)
            else:
                data_min = np.min(sample_points)
                data_max = np.max(sample_points)
                data_range = data_max - data_min
                
                if data_range > 0:
                    desired_absolute_bandwidth = data_range * 0.04
                else:
                    desired_absolute_bandwidth = 1e-5
                
                data_std = np.std(sample_points)
                forced_bw_method = desired_absolute_bandwidth / data_std if data_std > 0 else 'silverman'

                sns.kdeplot(
                    sample_points, 
                    lw=2, 
                    label=node, 
                    #bw_method=forced_bw_method, 
                    bw_adjust = 0.5,
                    gridsize=2000, 
                    cut=3, 
                    common_norm=False, 
                    ax=ax, 
                    color=node_color
                )

    ax.set_ylabel(' ')
    ax.legend(labels=[color_dict[node[:-3]][1] for node in nodes], fontsize=8)
    ax.set_xlim(left=0)

In [ ]:
# Probability distribution plots
dataset = 'comtrade'
Ps = [5.0, 25.0, 50.0]
num_samples = 20000
num_countries = 7
measure = 'all'
with_reserves = False
color_dict = color_dict_comtrade
num_ps = len(Ps)

# Row titles mapping
row_titles = [
    'Copper Ore',
    'Unrefined Copper',
    'Refined Copper',
    'Copper Wire'
]

# Create grid
fig, axes = plt.subplots(4, num_ps, figsize=(5 * num_ps, 16))

if num_ps == 1:
    axes = axes.reshape(4, 1)

means1 = pd.DataFrame()
means2 = pd.DataFrame()
means3 = pd.DataFrame()
means4 = pd.DataFrame()

# Loop through each P value and its corresponding column index
for col_idx, P in enumerate(Ps):
    
    with open(f'copper_{dataset}_{P}.pkl', 'rb') as f:
        res = pickle.load(f)

    all_base_changes = res['all_base_changes']
    all_tier_changes = res['tier_changes']

    test1 = all_tier_changes['Tier 1']
    test2 = all_tier_changes['Tier 2']
    test3 = all_tier_changes['Tier 3']
    test4 = all_tier_changes['Tier 4']

    all_tier_probs = res['agg_probs']
    all_edges = res['all_edges']
    tier_list = res['tier_list']

    if measure == 'base':
        countries_to_choose = all_edges.groupby('src_ref').sum().sort_values(by='change')['change']
        if not with_reserves:
            countries_to_choose = countries_to_choose[~countries_to_choose.index.str.contains(r'^R[A-Z]', regex=True)]
        
        country_list_1 = list(countries_to_choose[countries_to_choose.index.str.endswith('[3]')].index)
        country_list_2 = list(countries_to_choose[countries_to_choose.index.str.endswith('[2]')].index)
        country_list_3 = list(countries_to_choose[countries_to_choose.index.str.endswith('[1]')].index)
        country_list_4 = list(countries_to_choose[countries_to_choose.index.str.endswith('[0]')].index)
        
    elif measure == 'all':
        df_1, df_2 = copy(all_tier_changes['Tier 1']), copy(all_tier_changes['Tier 2'])
        df_3, df_4 = copy(all_tier_changes['Tier 3']), copy(all_tier_changes['Tier 4'])
        p_1, p_2 = copy(all_tier_probs['Tier 1']), copy(all_tier_probs['Tier 2'])
        p_3, p_4 = copy(all_tier_probs['Tier 3']), copy(all_tier_probs['Tier 4'])

        df_1['means'] = df_1.mul(p_1.iloc[0,:], axis=1).sum(axis=1)
        df_2['means'] = df_2.mul(p_2.iloc[0,:], axis=1).sum(axis=1)
        df_3['means'] = df_3.mul(p_3.iloc[0,:], axis=1).sum(axis=1)
        df_4['means'] = df_4.mul(p_4.iloc[0,:], axis=1).sum(axis=1)
        
        df_1 = df_1.sort_values(['means'], ascending=True)
        df_2 = df_2.sort_values(['means'], ascending=True)
        df_3 = df_3.sort_values(['means'], ascending=True)
        df_4 = df_4.sort_values(['means'], ascending=True)
        
        means1[col_idx] = copy(df_1['means'])
        means2[col_idx] = copy(df_2['means'])
        means3[col_idx] = copy(df_3['means'])
        means4[col_idx] = copy(df_4['means'])
        
        if not with_reserves:
            df_1 = df_1[~df_1.index.str.contains(r'^R[A-Z]', regex=True)]
            df_2 = df_2[~df_2.index.str.contains(r'^R[A-Z]', regex=True)]
            df_3 = df_3[~df_3.index.str.contains(r'^R[A-Z]', regex=True)]
            df_4 = df_4[~df_4.index.str.contains(r'^R[A-Z]', regex=True)]
            
        country_list_1 = list(df_1.index)
        country_list_2 = list(df_2.index)
        country_list_3 = list(df_3.index)
        country_list_4 = list(df_4.index)

    country_list_1 = country_list_1[:min(len(country_list_1), num_countries)]
    country_list_2 = country_list_2[:min(len(country_list_2), num_countries)]
    country_list_3 = country_list_3[:min(len(country_list_3), num_countries)]
    country_list_4 = country_list_4[:min(len(country_list_4), num_countries)]

    ax4, ax3, ax2, ax1 = axes[0, col_idx], axes[1, col_idx], axes[2, col_idx], axes[3, col_idx]

    # Plot distributions
    plot_less_likely_distributions(all_tier_changes, all_tier_probs, 'Tier 4', country_list_4, ax4, color_dict, num_samples)
    plot_less_likely_distributions(all_tier_changes, all_tier_probs, 'Tier 3', country_list_3, ax3, color_dict, num_samples)
    plot_less_likely_distributions(all_tier_changes, all_tier_probs, 'Tier 2', country_list_2, ax2, color_dict, num_samples)
    plot_less_likely_distributions(all_tier_changes, all_tier_probs, 'Tier 1', country_list_1, ax1, color_dict, num_samples)

    if col_idx > 0:
        for ax in [ax1, ax2, ax3, ax4]:
            ax.set_ylabel('')
            ax.tick_params(labelleft=False)

# Global axis layout adjustments
max_y = [1, 5, 15, 10]
for row_idx in range(4):
    row_axes = axes[row_idx, :]
    x_bounds = [ax.get_xlim() for ax in row_axes]
    row_max_x = max(bound[1] for bound in x_bounds)
    row_min_x = min(bound[0] for bound in x_bounds)
    
    for ax in row_axes:
        ax.set_xlim(row_min_x, row_max_x)
        ax.set_ylim(0, max_y[row_idx])

# Give subplots just enough breathing room at the top
fig.subplots_adjust(left=0.07, bottom=0.08, right=0.98, top=0.88, hspace=0.45, wspace=0.05)

# 1. Row headers: Position material names centered above each row's subplots
for row_idx in range(4):
    bbox = axes[row_idx, 0].get_position()
    header_y = bbox.y1 + 0.012
    
    fig.text(
        0.525, header_y, 
        row_titles[row_idx], 
        fontsize=13, 
        fontweight='bold', 
        ha='center', 
        va='bottom'
    )

# 2. Column headers: Position P-values safely above Row 0's "Copper Ore" label
row_0_top_y = axes[0, 0].get_position().y1
p_label_y = row_0_top_y + 0.038  # Sits neatly above the row title without overlapping

for col_idx, P in enumerate(Ps):
    col_bbox = axes[0, col_idx].get_position()
    col_center_x = (col_bbox.x0 + col_bbox.x1) / 2
    
    fig.text(
        col_center_x, p_label_y, 
        f'L = -{P}%', 
        fontsize=13, 
        fontweight='bold', 
        ha='center', 
        va='bottom'
    )

# 3. Axis Super Labels
fig.supxlabel('Average Monthly Losses (kTs)', y=0.03, fontsize=13)
fig.supylabel('Probability Density', x=0.02, fontsize=13)

plt.show()

In [ ]:
all_countries = np.unique(np.concat([country_list_1,country_list_2,country_list_3,country_list_4]))
import re
unique_plot_countries = sorted(list(set(re.sub(r'\[\d+\]', '', node) for node in all_countries)))
print(unique_plot_countries)

In [ ]:
#Table
top = 5
Ps = [5.0,25.0,50.0]
hs_code_list = [2603, 7402, 7403, 7408]
data_table = pd.DataFrame(np.zeros((top * len(Ps), 2 * len(hs_code_list))), dtype=object)
for i, P in enumerate(Ps):
    with open(f'copper_{dataset}_{P}.pkl', 'rb') as f:
        res = pickle.load(f)

    all_base_changes = res['all_base_changes']
    tier_1 = all_base_changes['Tier 1'].sort_values(ascending=True).head(top)
    tier_2 = all_base_changes['Tier 2'].sort_values(ascending=True).head(top)
    tier_3 = all_base_changes['Tier 3'].sort_values(ascending=True).head(top)
    tier_4 = all_base_changes['Tier 4'].sort_values(ascending=True).head(top)

    data_table.iloc[top*i:top*(i+1),6] = tier_1.index.str[:-3]
    data_table.iloc[top*i:top*(i+1),7] = tier_1.values/1e6
    data_table.iloc[top*i:top*(i+1),4] = tier_2.index.str[:-3]
    data_table.iloc[top*i:top*(i+1),5] = tier_2.values/1e6
    data_table.iloc[top*i:top*(i+1),2] = tier_3.index.str[:-3]
    data_table.iloc[top*i:top*(i+1),3] = tier_3.values/1e6
    data_table.iloc[top*i:top*(i+1),0] = tier_4.index.str[:-3]
    data_table.iloc[top*i:top*(i+1),1] = tier_4.values/1e6
    

In [ ]:
print(np.sum(df_1,0)[0])
print(np.sum(df_2,0)[0])
print(np.sum(df_3,0)[0])
print(np.sum(df_4,0)[0])

In [ ]:
print(np.sum(all_base_changes['Tier 1']))
print(np.sum(all_base_changes['Tier 2']))
print(np.sum(all_base_changes['Tier 3']))
print(np.sum(all_base_changes['Tier 4']))

In [ ]:
plot_base_scenarios(all_edges,'USA[4]',-P/100)